In [45]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, count, dense_rank, when, sum as Sum

In [2]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 20:09:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [28]:
trips = spark.read\
    .option("header","true")\
    .option("inferSchema","true")\
    .format("csv")\
    .load("files/trips.txt")

In [29]:
users = spark.read\
    .option("header","true")\
    .option("inferSchema","true")\
    .format("csv")\
    .load("files/users.txt")

In [30]:
trips.show()

+---+---------+---------+-------+-------------------+----------+
| id|client_id|driver_id|city_id|             status|request_at|
+---+---------+---------+-------+-------------------+----------+
|101|        1|       10|      1|          completed|2026-05-10|
|102|        2|       11|      1|cancelled_by_driver|2026-05-10|
|103|        3|       12|      6|          completed|2026-05-10|
|104|        4|       13|      1|          completed|2026-05-10|
|105|        1|       10|      6|cancelled_by_client|2026-05-11|
|106|        2|       11|      1|          completed|2026-05-11|
|107|        3|       12|     12|          completed|2026-05-11|
|108|        4|       10|     12|cancelled_by_client|2026-05-11|
|109|        1|       12|      1|          completed|2026-05-12|
|110|        3|       11|      6|          completed|2026-05-12|
|111|        2|       13|      1|cancelled_by_driver|2026-05-12|
|112|        4|       12|      6|          completed|2026-05-12|
|113|        1|       11|

In [31]:
users.show()

+--------+------+------+
|users_id|banned|  role|
+--------+------+------+
|       1|    No|client|
|       2|    No|client|
|       3|    No|client|
|       4|   Yes|client|
|      10|    No|driver|
|      11|    No|driver|
|      12|    No|driver|
|      13|   Yes|driver|
+--------+------+------+



In [39]:
unbanned_users = users.filter("banned='No'")

In [40]:
unbanned_users.show()

+--------+------+------+
|users_id|banned|  role|
+--------+------+------+
|       1|    No|client|
|       2|    No|client|
|       3|    No|client|
|      10|    No|driver|
|      11|    No|driver|
|      12|    No|driver|
+--------+------+------+



In [51]:
trips.filter("request_at between '2026-05-10' and '2026-05-12'")\
    .join(unbanned_users, on = (unbanned_users.users_id.isin(trips.client_id, trips.driver_id)), how="inner")\
    .withColumn("total_requests", count("id").over(Window.partitionBy("request_at")))\
    .withColumn("calc_2_sum", when(col("status").isin("cancelled_by_driver", "cancelled_by_client"), 1).otherwise(0))\
    .show()

+---+---------+---------+-------+-------------------+----------+--------+------+------+--------------+----------+
| id|client_id|driver_id|city_id|             status|request_at|users_id|banned|  role|total_requests|calc_2_sum|
+---+---------+---------+-------+-------------------+----------+--------+------+------+--------------+----------+
|101|        1|       10|      1|          completed|2026-05-10|       1|    No|client|             6|         0|
|101|        1|       10|      1|          completed|2026-05-10|      10|    No|driver|             6|         0|
|102|        2|       11|      1|cancelled_by_driver|2026-05-10|       2|    No|client|             6|         1|
|102|        2|       11|      1|cancelled_by_driver|2026-05-10|      11|    No|driver|             6|         1|
|103|        3|       12|      6|          completed|2026-05-10|       3|    No|client|             6|         0|
|103|        3|       12|      6|          completed|2026-05-10|      12|    No|driver| 

In [64]:
trips.filter("request_at between '2026-05-10' and '2026-05-12'")\
    .join(unbanned_users, on = (unbanned_users.users_id.isin(trips.client_id, trips.driver_id)), how="inner")\
    .withColumn("total_requests", count("id").over(Window.partitionBy("request_at")))\
    .groupBy("request_at", "total_requests").agg(
            Sum(
                   when(col("status").isin("cancelled_by_driver", "cancelled_by_client"), 1).otherwise(0)
               ).alias("summed_val"),
            (Sum(
                   when(col("status").isin("cancelled_by_driver", "cancelled_by_client"), 1).otherwise(0)
               )/col("total_requests")).alias("cancellation_rate")
    )\
.show()

+----------+--------------+----------+-------------------+
|request_at|total_requests|summed_val|  cancellation_rate|
+----------+--------------+----------+-------------------+
|2026-05-10|             6|         2| 0.3333333333333333|
|2026-05-11|             7|         3|0.42857142857142855|
|2026-05-12|             6|         1|0.16666666666666666|
+----------+--------------+----------+-------------------+

